### Model Integration with LLM Application (Groq, Google Gemini, OpenAI)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.7', 'langchain': '1.4.10'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CD40C326C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CD40D836E0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
# invoke - calling the model to execute the task and return the result
response = model.invoke("Hello How are you?")
response.content

'Hello! I’m doing great, thanks for asking. How can I help you today?'

For Using a specific LLM

In [4]:
"""
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
response = agent.invoke({"messages":[{"role":"user","content":"What is the weather in New York?"}]})
print(response.content) 
"""


'\nfrom langchain_google_genai import ChatGoogleGenerativeAI\nmodel = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")\nresponse = agent.invoke({"messages":[{"role":"user","content":"What is the weather in New York?"}]})\nprint(response.content) \n'

### Streaming and Batch
#### Streaming
- It significally improves user experience for particularly longer responses. It allows you to receive model outputs chunk-by-chunk or token-by-token in real time as the AI generates them
- Rather than waiting for the entire response to finish before displaying it

In [5]:
model.invoke("Write a 200 word paragraph on Machine Learning")

AIMessage(content='Machine learning, a subfield of artificial intelligence, empowers computers to improve performance on tasks through experience rather than explicit programming, relying on algorithms that detect patterns within data and adjust their internal parameters accordingly. By ingesting vast datasets, supervised models learn mappings from inputs to desired outputs, while unsupervised techniques uncover hidden structures without labeled examples, and reinforcement methods train agents to maximize cumulative rewards through trial and error. Applications span diverse domains: predictive maintenance in manufacturing, personalized recommendations in e‑commerce, medical diagnosis assistance, autonomous navigation, and natural language processing that powers chatbots and translation services. Core concepts such as feature engineering, model selection, regularization, and validation ensure that learned representations generalize beyond training examples and avoid overfitting. Recent 

In [6]:
model.stream("Write a 200 word paragraph on Machine Learning") 
# <generator object BaseChatModel.stream at 0x000001D68C558420> - means it is ready to stream and hasn't been consumed it

<generator object BaseChatModel.stream at 0x000001CD408E9780>

In [7]:
for chunk in model.stream("Write a 200 word paragraph on Machine Learning"):
    print(chunk.text, end='',flush=True)

# chunk.text - extracts the text from the chunk
# flush - displays output immediately 

Machine learning, a subfield of artificial intelligence, empowers computers to improve performance on tasks through experience rather than explicit programming, leveraging algorithms that detect patterns within data and adapt their models accordingly. Supervised learning trains models on labeled examples, enabling tasks such as image classification, speech recognition, and medical diagnosis by mapping inputs to known outputs; unsupervised learning discovers hidden structures in unlabeled data, supporting clustering, dimensionality reduction, and anomaly detection. Reinforcement learning, meanwhile, teaches agents to maximize cumulative reward through trial‑and‑error interaction with environments, exemplified by game‑playing systems and autonomous robotics. Core techniques include linear regression, decision trees, support vector machines, neural networks, and learning architectures that stack multiple layers to capture hierarchical representations, dramatically advancing fields like na

#### Batch
It is a collection of independent requests to a model can significally improve performance and reduce costs as the processing be done in parallel

In [8]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "Who invented zero?",
    "What is Quantum Computing?"],
    config = { 
        'max_concurrency': 5, # Limit to 5 parallel calls
    }
)
# we use max_concurreny - to prevent rate limits (HTTP 429 Errors - Too many requests) and avoid resouce exhaustion

for response in responses:
    print(response)

content='Parrots are famous for their bright, multi‑coloured plumage, and scientists think the colors serve several, often overlapping, purposes that have been shaped by evolution:\n\n| Function | How it works | Why it matters for parrots |\n|----------|--------------|---------------------------|\n| **Sexual selection / mate choice** | Bright, saturated colours (especially reds, oranges, yellows) are produced by pigments that are costly to acquire or synthesize. Individuals that can display the most vivid hues are often healthier or better at finding food, so they become more attractive to potential mates. | In many parrot species the pair bond is long‑term, so choosing a high‑quality partner early on improves the chances of raising successful offspring. |\n| **Species and individual recognition** | Specific colour patterns (e.g., the blue head of a blue‑and‑gold macaw vs. the green head of a scarlet macaw) act like visual “name tags.” | In dense tropical forests, visual cues help bird